<a href="https://colab.research.google.com/github/nghff/vlm-pca-exercise/blob/main/Raphi_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 167.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.modules import padding
from torch.utils.data._utils.collate import default_collate

from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
from PIL import Image, UnidentifiedImageError

import io, os, base64, requests
import regex as re
from typing import List
import numpy as np
import pandas as pd
from hashlib import sha256
import sys

import time
from tqdm import tqdm as tqdm

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/'
SAVE_DIR = os.path.join(DRIVE_DIR, 'MyDrive/Raphi_Task_Data')
if not os.path.exists(SAVE_DIR):
  os.makedirs(SAVE_DIR)

DATA_PORTION = 1

model_id = "Qwen/Qwen3-VL-2B-Instruct"
#model_id = "llava-hf/llava-1.5-7b-hf"
#model_id = "allenai/Molmo2-8B"

RESULTS_SAVE_DIR = os.path.join(SAVE_DIR, model_id)
if not os.path.exists(RESULTS_SAVE_DIR):
  os.makedirs(RESULTS_SAVE_DIR)

Mounted at /content/drive


In [ ]:
def save_df(name, df, show_df=False):
    df_path = os.path.join(RESULTS_SAVE_DIR, f'{name}.pkl')
    df.to_pickle(df_path)

    if show_df:
        display(df)

def load_df(name):
    df_path = os.path.join(RESULTS_SAVE_DIR, f'{name}.pkl')
    return pd.read_pickle(df_path)

### Dataset implementation for pixmo dataset

In [ ]:
def pixmo_collate_fn(processor, timeout=10, max_length: int = 4096):
    def collate_fn(batch):
        messages_list = [b["messages"] for b in batch]
        targets = torch.tensor([b["target_count"] for b in batch])
        bad_messages_idx = []

        for i in range(len(messages_list)):
            for message in messages_list[i]:
                if message["role"] != "user":
                    continue
                for content in message["content"]:
                    if content["type"] != "image":
                        continue
                    img_url = content["image"]

                    if img_url.startswith("http://") or img_url.startswith("https://"):
                        try:
                            r = requests.get(img_url, timeout=timeout, allow_redirects=True)
                            r.raise_for_status()
                        except:
                            print(f"request failed: {img_url}")
                            bad_messages_idx.append(i)
                            break
                        ct = (r.headers.get("content-type") or "").lower()

                        if "image" not in ct:
                            print(f"filtered: {img_url}")
                            bad_messages_idx.append(i)
                            break
                    else:
                        print(f"filtered: {img_url}")
                        bad_messages_idx.append(i)
                        break
                break

        # remove messages with failed images
        for idx in sorted(bad_messages_idx, reverse=True):
            targets = torch.cat([targets[:idx], targets[idx+1:]])
            del messages_list[idx]

        print(f'num messages: {len(messages_list)}')

        inputs = processor.apply_chat_template(
            messages_list,
            tokenize=True,
            continue_final_message=True,
            return_dict=True,
            return_tensors="pt",
            padding='longest'
        )

        inputs["target_count"] = targets
        return inputs
    return collate_fn

class pixmoDataset(Dataset):
  splits = {
    'val': 'validation-00000-of-00001.parquet',
    'train': 'train-00000-of-00001.parquet',
    'test': 'test-00000-of-00001.parquet'
  }

  def __init__(self,
               split: str,
               processor: AutoProcessor,
               portion=1.0, max_length=4096,
               **kwargs):
    if split == "validation":
      split = "val"
    if split not in pixmoDataset.splits:
      raise ValueError("'split' not recognized in dataset constructor")

    split_file = pixmoDataset.splits[split]
    self.data = pd.read_parquet("hf://datasets/allenai/pixmo-count/data/" + split_file)
    self.data = self.data.filter(items=['image_url', 'label', 'count'])
    self.processor = processor
    self.portion = portion
    self.max_length = max_length
    if 'filter_fn' in kwargs:
      self.data = kwargs['filter_fn'](self.data)

  def __len__(self):
    return int(len(self.data) * self.portion)

  def __getitem__(self, idx):
        row = self.data.iloc[idx]
        label = row["label"]
        image = row["image_url"]
        target = int(row["count"])

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": f"How many {label} are in this photo?"},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": f"The number of {label} is: "}],
            },
        ]

        return {"messages": messages, "target_count": target}

### Evaluator

In [ ]:
# define metrics
def sanitize_pairs(pred: List[str], target: List[str]):
  p = []
  t = []
  misfits = []
  for i in range(len(pred)):
    if pred[i] != '':
      p.append(int(re.sub(r'[^\d]+', '', pred[i])))
      t.append(target[i])
    else:
      misfits.append((pred[i], target[i]))
  return p, t, misfits

def accuracy(pred: List[int], target: List[int]):
  return np.mean(pred == target)

def mean_error(pred: List[int], target: List[int]):
  return np.mean(np.abs(pred - target))

def mean_squared_error(pred: List[int], target: List[int]):
  return np.mean((pred - target)**2)

In [ ]:
class Evaluator:
  def __init__(self,
               metric_fns={},
               batch_size=4,
               shuffle=False,
               collate_fn=None,
               token_allowance=2,
               callbacks=[]):
    self.metric_fns = metric_fns
    self.batch_size = batch_size
    self.shuffle = shuffle
    self.collate_fn = collate_fn
    self.max_new_tokens = token_allowance
    self.callbacks = callbacks

  def __call__(self, model, processor, dataset, **kwargs):
    return self.evaluate(model, processor, dataset, **kwargs)

  def evaluate(self, model, processor, dataset, **kwargs):
    dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=self.shuffle, collate_fn=self.collate_fn)
    all_preds = []
    all_targets = []
    all_inputs = []
    all_misfits = []
    dl_progress = tqdm(dataloader)

    model.eval()
    with torch.inference_mode():
        for batch in dl_progress:
            target_counts = batch.pop('target_count')
            target_counts = [count.item() for count in target_counts]

            inputs = batch
            inputs = {k: v.to(model.device, non_blocking=True) for k, v in inputs.items()}

            output = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                num_beams=1,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=False,
                output_hidden_states=kwargs.get('needs_hidden_states', False),
                output_attentions=kwargs.get('needs_attentions', False)
            )

            generated_ids_trimmed = [
                out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs['input_ids'], output['sequences'])
            ]
            generated_strs = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )

            generated_preds, target_counts, misfits = sanitize_pairs(generated_strs, target_counts)

            all_preds.extend(generated_preds)
            all_targets.extend(target_counts)
            all_inputs.extend([input_ids for input_ids in inputs['input_ids']])
            all_misfits.extend(misfits)

            print(f'preds: {generated_preds}')
            print(f'targets: {target_counts}')
            for callback in self.callbacks:
                callback(
                    inputs,
                    target_counts,
                    generated_preds,
                    output['hidden_states'] if 'hidden_states' in output else None,
                    output['attentions'] if 'attentions' in output else None
                )

    metrics = {}
    for name, fn in metric_fns.items():
        metrics[name] = fn(np.array(all_preds), np.array(all_targets))

    return {
        'metrics': metrics,
        'all_inputs': all_inputs,
        'all_preds': all_preds,
        'all_targets': all_targets,
        'all_misfits': all_misfits
    }

## Evaluation

### Run evaluation

In [ ]:
load_params = {
    'Qwen/Qwen3-VL-2B-Instruct': {
        'dtype': torch.bfloat16,
        'device_map': "auto"
    },

    'llava-hf/llava-1.5-7b-hf': {
        'torch_dtype': torch.float16,
        'low_cpu_mem_usage': True
    },

    'allenai/Molmo2-8B': {
        'dtype': "auto",
        'device_map': "auto",
        'trust_remote_code': True
    }
}

model = AutoModelForImageTextToText.from_pretrained(
     model_id, **load_params[model_id]
)

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code = True)
processor.tokenizer.padding_side = "left"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
test_dataset = pixmoDataset(split='test', processor=processor, portion=DATA_PORTION)
print(len(test_dataset))

metric_fns = {
    "accuracy": accuracy,
    "mean_error": mean_error,
    "mean_squared_error": mean_squared_error
}

540


In [ ]:
evaluator = Evaluator(
    metric_fns=metric_fns,
    batch_size=4,
    shuffle=False,
    collate_fn=pixmo_collate_fn(processor)
)
results = evaluator(model, processor, test_dataset)

# show misfits
misfits_df = pd.DataFrame(results['all_misfits'], columns=['pred', 'target'])
save_df('all_misfits', misfits_df, show_df=True)

# display results
results_df = pd.DataFrame(results['metrics'], index=[0])
save_df('results', results_df, show_df=True)
results_df

# save all input-pred-target pairs
all_df = pd.DataFrame(zip(results['all_inputs'], results['all_preds'], results['all_targets']), columns=['input', 'pred', 'target'])
save_df('all_input_pred_target_pairs', all_df, show_df=False)

  0%|          | 0/135 [00:00<?, ?it/s]

num messages: 4


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  1%|          | 1/135 [00:23<52:28, 23.50s/it]

preds: [5, 9, 9, 4]
targets: [5, 10, 10, 4]
num messages: 4


  1%|▏         | 2/135 [00:59<1:08:54, 31.08s/it]

...

preds: [4, 4, 2, 4]
targets: [4, 4, 2, 4]
num messages: 4


100%|██████████| 135/135 [1:19:21<00:00, 35.27s/it]

preds: [2, 6, 4, 4]
targets: [2, 8, 4, 4]


,pred,target


,accuracy,mean_error,mean_squared_error
0,0.628083,0.565465,1.392789


## PCA Analysis

In [ ]:
load_params = {
    'Qwen/Qwen3-VL-2B-Instruct': {
        'dtype': torch.bfloat16,
        'device_map': "auto",
        'attn_implementation': 'eager'
    },

    'llava-hf/llava-1.5-7b-hf': {
        'torch_dtype': torch.float16,
        'low_cpu_mem_usage': True,
        'attn_implementation': 'eager'
    },

    'allenai/Molmo2-8B': {
        'dtype': "auto",
        'device_map': "auto",
        'trust_remote_code': True
    }
}

model = AutoModelForImageTextToText.from_pretrained(
     model_id, **load_params[model_id]
).to('cuda:0')
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code = True)
processor.tokenizer.padding_side = "left"

config.json: 0.00B [00:00, ?B/s]

configuration_molmo2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/allenai/Molmo2-8B:
- configuration_molmo2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_molmo2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/allenai/Molmo2-8B:
- modeling_molmo2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.5M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

### Logging

In [ ]:
layers_to_hook = {
    'Qwen/Qwen3-VL-2B-Instruct': [
        'model.visual.blocks.1',                # early visual features extracted by visual encoder
        'model.visual.blocks.11',                # mid visual features extracted by visual encoder
        'model.visual.blocks.23',               # final visual information extracted from image by visual encoder

        'model.visual.merger',                  # visual features 'mapped' to global language space

        'model.visual.deepstack_merger_list.0', # visual features added to corresponding activations during early language reasoning (in the 'language' decoder)
        'model.visual.deepstack_merger_list.1', # visual features added to corresponding activations during early language reasoning (in the 'language' decoder)
        'model.visual.deepstack_merger_list.2', # visual features added to corresponding activations during early language reasoning (in the 'language' decoder)

        'model.language_model.layers.0',
        'model.language_model.layers.1',        # early language features extracted by decoder
        'model.language_model.layers.13',       # mid language features extracted by decoder
        'model.language_model.layers.20',
        'model.language_model.layers.27'       # final language features extracted by decoder
    ],
    'llava-hf/llava-1.5-7b-hf': [
        'model.vision_tower.vision_model.encoder.layers.1',
        'model.vision_tower.vision_model.encoder.layers.11',
        'model.vision_tower.vision_model.encoder.layers.23',

        'model.multi_modal_projector',

        'model.language_model.layers.0',
        'model.language_model.layers.1',
        'model.language_model.layers.15',
        'model.language_model.layers.23',
        'model.language_model.layers.31'
    ],
    'allenai/Molmo2-8B':[
        'model.vision_backbone.image_vit.transformer.resblocks.1',
        'model.vision_backbone.image_vit.transformer.resblocks.12',
        'model.vision_backbone.image_vit.transformer.resblocks.24',

        'model.vision_backbone.image_projector',

        'model.transformer.blocks.0',
        'model.transformer.blocks.1',
        'model.transformer.blocks.17',
        'model.transformer.blocks.26',
        'model.transformer.blocks.35'
    ]
}

attentions_to_grab = {
    'Qwen/Qwen3-VL-2B-Instruct': [
        0, 1, 13, 27
    ],
    'llava-hf/llava-1.5-7b-hf': [
        0, 1, 15, 30, 31
    ],
    'allenai/Molmo2-8B':[
        # eager attention not supported
    ]
}

In [ ]:
# activation cache
activation_cache = {}

# activation data
activation_data = []

# hook for activation logging
def log_activations(module_name):
    def hook(module, inp, out):
        print(f'caught {module_name}')
        x = out[0] if isinstance(out, (tuple, list)) else out
        if torch.is_tensor(x):
            activation_cache[module_name] = (
                x.detach()
                .to(dtype=torch.float32)
                .cpu()
            )
    return hook

# attach hooks
def attach_hooks(model, model_id):
    to_hook = layers_to_hook[model_id]
    for module_name, module in model.named_modules():
        if module_name in to_hook:
            print(f'hooked: {module_name}')
            module.register_forward_hook(log_activations(module_name))

# callback for storing activation data
def store_activations(inputs, targets, preds, hidden_states, attentions):
    print('detaching activations')
    attentions = attentions[0] # assumes single token generated

    input_ids_cpu = inputs["input_ids"].detach().cpu().tolist()
    tokens_batch = [processor.tokenizer.convert_ids_to_tokens(ids) for ids in input_ids_cpu]

    act_cpu = {layer: activation_cache[layer].detach().cpu().numpy() for layer in layers_to_hook[model_id]}

    att_cpu = {lid: attentions[lid].detach().cpu().to(torch.float16).numpy() for lid in attentions_to_grab[model_id]}

    print('detached data, storing')
    for i in range(len(targets)):
        print(f'storing idx {i} in batch')
        activation_data.append({
            'input_tokens': tokens_batch[i],
            'target': targets[i],
            'pred': preds[i]
        })
        for layer in layers_to_hook[model_id]:
            print(f'storing {layer}')
            activation_data[-1][layer] = act_cpu[layer][i]

        if attentions_to_grab[model_id] != []:
            for layer_id in attentions_to_grab[model_id]:
                print(f'storing attention_{layer_id}')
                activation_data[-1][f'attention_{layer_id}'] = att_cpu[layer_id][i].mean(axis=-3)

# clear hooks
def clear_hooks():
    for module_name, module in model.named_modules():
        if module_name in layers_to_hook[model_id]:
            module._forward_hooks.clear()

# reset logs
def reset_logs():
    activation_cache.clear()
    activation_data.clear()
    clear_hooks()

# pca plotting
def layer_pca_plots():
  pass

### Run feedforward

In [ ]:
# function for filtering out rows in pixmo with count > 5

def counts_filter(data: pd.DataFrame):
    return data.iloc[(data['count'].to_numpy() <= 5).tolist()].reset_index(drop=True)

In [ ]:
test_dataset = pixmoDataset(split='test', processor=processor, portion=DATA_PORTION, filter_fn=counts_filter)

print(f'test dataset size: {len(test_dataset)}')
print(test_dataset.data.head(10))

metric_fns = {}

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 164299ad-58ab-4b6b-b264-1078a3d72ba6)')' thrown while requesting GET https://huggingface.co/datasets/allenai/pixmo-count/resolve/main/data/test-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].


test dataset size: 240
                                           image_url        label  count
0  https://live.staticflickr.com/65535/5072578687...       people      5
1  https://live.staticflickr.com/3672/9624035497_...       people      4
2  https://live.staticflickr.com/1979/45001822892...       people      5
3  https://live.staticflickr.com/3692/9251404684_...       people      4
4  https://live.staticflickr.com/7509/16135146130...       people      3
5  https://live.staticflickr.com/2278/2077685668_...         dogs      3
6  https://live.staticflickr.com/7427/9935572953_...  helicopters      5
7  https://live.staticflickr.com/7350/27704496056...        ducks      3
8  https://live.staticflickr.com/7076/7168922099_...       people      2
9  https://live.staticflickr.com/2918/14390837392...    airplanes      5


In [ ]:
reset_logs()

evaluator = Evaluator(
    metric_fns=metric_fns,
    batch_size=4,
    shuffle=False,
    collate_fn=pixmo_collate_fn(processor),
    token_allowance=1, # 1 thru 9 represented by uno token,
    callbacks=[store_activations]
)

attach_hooks(model, model_id)

results = evaluator(model, processor, test_dataset, needs_attentions=True)

save_df('activation_data', pd.DataFrame(activation_data), show_df=True)

clear_hooks()

hooked: model.transformer.blocks.0
hooked: model.transformer.blocks.1
hooked: model.transformer.blocks.17
hooked: model.transformer.blocks.26
hooked: model.transformer.blocks.35
hooked: model.vision_backbone.image_vit.transformer.resblocks.1
hooked: model.vision_backbone.image_vit.transformer.resblocks.12
hooked: model.vision_backbone.image_vit.transformer.resblocks.24
hooked: model.vision_backbone.image_projector


  0%|          | 0/60 [00:00<?, ?it/s]

num messages: 4
caught model.vision_backbone.image_vit.transformer.resblocks.1
caught model.vision_backbone.image_vit.transformer.resblocks.12
caught model.vision_backbone.image_vit.transformer.resblocks.24
caught model.vision_backbone.image_projector
caught model.transformer.blocks.0
caught model.transformer.blocks.1
caught model.transformer.blocks.17
caught model.transformer.blocks.26
caught model.transformer.blocks.35


  2%|▏         | 1/60 [00:06<06:04,  6.17s/it]

preds: [5, 4, 5, 4]
targets: [5, 4, 5, 4]
detaching activations
detached data, storing
storing idx 0 in batch
storing model.vision_backbone.image_vit.transformer.resblocks.1
storing model.vision_backbone.image_vit.transformer.resblocks.12
storing model.vision_backbone.image_vit.transformer.resblocks.24
storing model.vision_backbone.image_projector
storing model.transformer.blocks.0
storing model.transformer.blocks.1
storing model.transformer.blocks.17
storing model.transformer.blocks.26
storing model.transformer.blocks.35
storing idx 1 in batch
storing model.vision_backbone.image_vit.transformer.resblocks.1
storing model.vision_backbone.image_vit.transformer.resblocks.12
storing model.vision_backbone.image_vit.transformer.resblocks.24
storing model.vision_backbone.image_projector
storing model.transformer.blocks.0
storing model.transformer.blocks.1
storing model.transformer.blocks.17
storing model.transformer.blocks.26
storing model.transformer.blocks.35
storing idx 2 in batch
storing 

...

 98%|█████████▊| 59/60 [06:01<00:06,  6.13s/it]

preds: [5, 4, 4, 2]
targets: [5, 4, 4, 2]
detaching activations
detached data, storing
storing idx 0 in batch
storing model.vision_backbone.image_vit.transformer.resblocks.1
storing model.vision_backbone.image_vit.transformer.resblocks.12
storing model.vision_backbone.image_vit.transformer.resblocks.24
storing model.vision_backbone.image_projector
storing model.transformer.blocks.0
storing model.transformer.blocks.1
storing model.transformer.blocks.17
storing model.transformer.blocks.26
storing model.transformer.blocks.35
storing idx 1 in batch
storing model.vision_backbone.image_vit.transformer.resblocks.1
storing model.vision_backbone.image_vit.transformer.resblocks.12
storing model.vision_backbone.image_vit.transformer.resblocks.24
storing model.vision_backbone.image_projector
storing model.transformer.blocks.0
storing model.transformer.blocks.1
storing model.transformer.blocks.17
storing model.transformer.blocks.26
storing model.transformer.blocks.35
storing idx 2 in batch
storing 

100%|██████████| 60/60 [06:07<00:00,  6.13s/it]

preds: [4, 2, 4, 4]
targets: [4, 2, 4, 4]
detaching activations
detached data, storing
storing idx 0 in batch
storing model.vision_backbone.image_vit.transformer.resblocks.1
storing model.vision_backbone.image_vit.transformer.resblocks.12
storing model.vision_backbone.image_vit.transformer.resblocks.24
storing model.vision_backbone.image_projector
storing model.transformer.blocks.0
storing model.transformer.blocks.1
storing model.transformer.blocks.17
storing model.transformer.blocks.26
storing model.transformer.blocks.35
storing idx 1 in batch
storing model.vision_backbone.image_vit.transformer.resblocks.1
storing model.vision_backbone.image_vit.transformer.resblocks.12
storing model.vision_backbone.image_vit.transformer.resblocks.24
storing model.vision_backbone.image_projector
storing model.transformer.blocks.0
storing model.transformer.blocks.1
storing model.transformer.blocks.17
storing model.transformer.blocks.26
storing model.transformer.blocks.35
storing idx 2 in batch
storing 

,input_tokens,target,pred,model.vision_backbone.image_vit.transformer.resblocks.1,model.vision_backbone.image_vit.transformer.resblocks.12,model.vision_backbone.image_vit.transformer.resblocks.24,model.vision_backbone.image_projector,model.transformer.blocks.0,model.transformer.blocks.1,model.transformer.blocks.17,model.transformer.blocks.26,model.transformer.blocks.35
0,"[<|im_end|>, <low_res_im_start>, <im_patch>, <...",5,5,"[[0.09089747, 0.094417684, 0.0795076, -0.14841...","[[-0.05144971, -0.056074128, 0.038487792, -0.1...","[[-0.8997775, 0.9769423, -0.298482, 0.48140284...","[[-1.2173767, 0.5716996, 0.42625633, -0.320909...","[[-0.051880065, 0.042939056, -0.03975417, -0.3...","[[-0.10301135, 0.109460935, -0.06389668, -0.32...","[[3.7860124, 3.9851258, -2.5295005, -9.458545,...","[[1.6375004, 3.9255838, -3.1217768, -8.817377,...","[[109.70213, 5.5074663, 133.76143, -47.16627, ..."
1,"[<|endoftext|>, <|endoftext|>, <|endoftext|>, ...",4,4,"[[-0.00074416026, 0.16850665, -0.009863712, -0...","[[-0.16084145, -0.16606763, -0.18470135, 0.089...","[[-0.18915856, 0.08319497, -0.5412944, 0.10525...","[[-0.67370325, 0.27498043, 2.6290443, 1.806113...","[[0.42189744, 0.097749785, 1.0772955, 0.131759...","[[0.6224927, 0.063491434, 1.0132124, 0.1249691...","[[-0.22283438, 0.72549707, -1.1144251, 0.23154...","[[-2.2662284, -1.9191232, 1.6694223, 0.4329958...","[[11.907283, -1.8353949, 12.850586, -12.401773..."
2,"[<|im_end|>, <low_res_im_start>, <im_patch>, <...",5,5,"[[0.014729757, 0.051083386, 0.08793838, -0.084...","[[-0.043284364, -0.06413045, -0.08127661, -0.0...","[[-0.9932815, -0.1285617, -0.30303812, 0.60863...","[[-0.5621706, 0.5748884, -0.025511047, -0.8323...","[[-0.051880065, 0.042939056, -0.03975417, -0.3...","[[-0.10301135, 0.109460935, -0.06389668, -0.32...","[[3.7860124, 3.9851258, -2.5295005, -9.458545,...","[[1.6375004, 3.9255838, -3.1217768, -8.817377,...","[[109.70213, 5.5074663, 133.76143, -47.16627, ..."
3,"[<|im_end|>, <low_res_im_start>, <im_patch>, <...",4,4,"[[0.01979322, 0.046457022, 0.048576266, -0.158...","[[0.3621316, 0.0018665642, -0.68047124, 0.2807...","[[0.36855394, 0.1423938, -0.99600714, 0.865866...","[[-0.7992014, 0.88124514, -1.5055662, -0.34945...","[[-0.051880065, 0.042939056, -0.03975417, -0.3...","[[-0.10301135, 0.109460935, -0.06389668, -0.32...","[[3.7860124, 3.9851258, -2.5295005, -9.458545,...","[[1.6375004, 3.9255838, -3.1217768, -8.817377,...","[[109.70213, 5.5074663, 133.76143, -47.16627, ..."
4,"[<|endoftext|>, <|endoftext|>, <|endoftext|>, ...",3,3,"[[0.11112512, -0.08760736, 0.2911582, -0.14200...","[[0.48816156, -0.2540673, 0.19036978, -0.02865...","[[-1.0807502, 0.099523306, -0.14507145, 0.1875...","[[-0.83966154, 0.020146454, 0.6133335, -0.3820...","[[0.42189744, 0.097749785, 1.0772955, 0.131759...","[[0.6224927, 0.063491434, 1.0132124, 0.1249691...","[[-0.22283438, 0.72549707, -1.1144251, 0.23154...","[[-2.2662284, -1.9191232, 1.6694223, 0.4329958...","[[11.907283, -1.8353949, 12.850586, -12.401773..."
...,...,...,...,...,...,...,...,...,...,...,...,...
234,"[<|im_end|>, <low_res_im_start>, <im_patch>, <...",2,2,"[[0.17610297, 0.336736, 0.33519712, -0.1280908...","[[-0.12851481, 0.111093655, 0.12337954, 0.2932...","[[0.090909004, -0.18073854, -0.03732586, 0.952...","[[-0.89622533, -0.07364055, 0.14212687, 0.9931...","[[-0.051880065, 0.042939056, -0.03975417, -0.3...","[[-0.10301135, 0.109460935, -0.06389668, -0.32...","[[3.7860124, 3.9851258, -2.5295005, -9.458545,...","[[1.6375004, 3.9255838, -3.1217768, -8.817377,...","[[109.70213, 5.5074663, 133.76143, -47.16627, ..."
235,"[<|endoftext|>, <|endoftext|>, <|endoftext|>, ...",4,4,"[[-0.011503331, 0.049438305, 0.08440389, -0.07...","[[-0.011396177, -0.14299646, 0.2644027, -0.060...","[[0.048731804, 0.17382777, 0.32908922, 0.50340...","[[-0.7176744, -1.0352402, -0.72593063, -0.9346...","[[0.42189744, 0.097749785, 1.0772955, 0.131759...","[[0.6224927, 0.063491434, 1.0132124, 0.1249691...","[[-0.22283438, 0.72549707, -1.1144251, 0.2315

In [ ]:
pd.DataFrame(activation_data).head()

In [ ]:

print('a')
load_df('activation_data')['attention_1'][0]

In [ ]:
activation_data[0]['attention_1'].shape

In [ ]:
#show
plt.imshow(activation_data[0]['attention_1'], vmin=0, vmax=0.01)

In [ ]:
for name, module in model.model.transformer.blocks[0].self_attn.k_norm.named_modules():
    print(name)

In [ ]:
print(len([name for name, module in model.named_modules()]))

819


In [ ]:
for name, module in model.named_modules():
  print(name)


model
model.transformer
model.transformer.wte
model.transformer.emb_drop
model.transformer.blocks
model.transformer.blocks.0
model.transformer.blocks.0.self_attn
model.transformer.blocks.0.self_attn.att_proj
model.transformer.blocks.0.self_attn.k_norm
model.transformer.blocks.0.self_attn.q_norm
model.transformer.blocks.0.self_attn.attn_out
model.transformer.blocks.0.attn_norm
model.transformer.blocks.0.dropout
model.transformer.blocks.0.mlp
model.transformer.blocks.0.mlp.ff_proj
model.transformer.blocks.0.mlp.ff_out
model.transformer.blocks.0.mlp.act
model.transformer.blocks.0.ff_norm
model.transformer.blocks.1
model.transformer.blocks.1.self_attn
model.transformer.blocks.1.self_attn.att_proj
model.transformer.blocks.1.self_attn.k_norm
model.transformer.blocks.1.self_attn.q_norm
model.transformer.blocks.1.self_attn.attn_out
model.transformer.blocks.1.attn_norm
model.transformer.blocks.1.dropout
model.transformer.blocks.1.mlp
model.transformer.blocks.1.mlp.ff_proj
model.transformer.blo

In [ ]:
model

Molmo2ForConditionalGeneration(
  (model): Molmo2Model(
    (transformer): Molmo2TextModel(
      (wte): Molmo2Embedding()
      (emb_drop): Dropout(p=0.0, inplace=False)
      (blocks): ModuleList(
        (0-35): 36 x Molmo2DecoderLayer(
          (self_attn): Molmo2Attention(
            (att_proj): Linear(in_features=4096, out_features=6144, bias=False)
            (k_norm): Molmo2RMSNorm((128,), eps=1e-06)
            (q_norm): Molmo2RMSNorm((128,), eps=1e-06)
            (attn_out): Linear(in_features=4096, out_features=4096, bias=False)
          )
          (attn_norm): Molmo2RMSNorm((4096,), eps=1e-06)
          (dropout): Dropout(p=0.0, inplace=False)
          (mlp): LanguageModelMLP(
            (ff_proj): Linear(in_features=4096, out_features=24576, bias=False)
            (ff_out): Linear(in_features=12288, out_features=4096, bias=False)
            (act): SiLUActivation()
          )
          (ff_norm): Molmo2RMSNorm((4096,), eps=1e-06)
        )
      )
      (ln_f): M